In [1]:
!nvidia-smi

Mon Apr  6 20:35:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [11]:
!pip install mpi4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.6 MB/s  0:00:00


In [2]:
!pip install -U pip
!pip install torch transformers>=4.43.0 datasets accelerate deepspeed huggingface_hub sentencepiece safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [3]:
%%writefile ds_zero2.json
{
  "train_micro_batch_size_per_gpu": "auto",
  "gradient_accumulation_steps": "auto",
  "gradient_clipping": 1.0,
  "bf16": {
    "enabled": "auto"
  },
  "fp16": {
    "enabled": false
  },
  "zero_optimization": {
    "stage": 2,
    "allgather_partitions": true,
    "allgather_bucket_size": 200000000,
    "reduce_scatter": true,
    "reduce_bucket_size": 200000000,
    "overlap_comm": true,
    "contiguous_gradients": true
  },
  "steps_per_print": 50,
  "wall_clock_breakdown": false
}

Writing ds_zero2.json


In [14]:
%%writefile train_zero2_llama.py
import os
import time
import json
import argparse
from typing import Dict, List, Any

import torch
import torch.distributed as dist

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed,
)


def is_main_process() -> bool:
    return int(os.environ.get("RANK", "0")) == 0


def reduce_sum(value: int, device: torch.device) -> int:
    if dist.is_available() and dist.is_initialized():
        t = torch.tensor([value], device=device, dtype=torch.long)
        dist.all_reduce(t, op=dist.ReduceOp.SUM)
        return int(t.item())
    return int(value)


def format_messages_as_text(messages: List[Dict[str, str]], eos_token: str) -> str:
    parts = []
    for msg in messages:
        role = msg["role"].strip().lower()
        content = msg["content"].strip()

        if role == "user":
            prefix = "User:"
        elif role == "assistant":
            prefix = "Assistant:"
        else:
            prefix = f"{role.capitalize()}:"

        parts.append(f"{prefix}\n{content}")

    return "\n\n".join(parts) + eos_token


class TokenCountingTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.window_tokens = 0
        self.window_start = time.time()

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        if model.training:
            if "attention_mask" in inputs:
                local_tokens = int(inputs["attention_mask"].sum().item())
            else:
                local_tokens = int(inputs["input_ids"].numel())

            device = inputs["input_ids"].device
            global_tokens = reduce_sum(local_tokens, device)
            self.window_tokens += global_tokens

        return super().compute_loss(
            model,
            inputs,
            return_outputs=return_outputs,
            num_items_in_batch=num_items_in_batch,
        )

    def log(self, logs: Dict[str, float], start_time: float = None) -> None:
        now = time.time()
        elapsed = max(now - self.window_start, 1e-8)

        if self.window_tokens > 0:
            logs["tokens_per_sec"] = self.window_tokens / elapsed

        if torch.cuda.is_available():
            logs["gpu_peak_mem_gb"] = torch.cuda.max_memory_allocated() / (1024 ** 3)

        super().log(logs, start_time=start_time)

        self.window_tokens = 0
        self.window_start = now


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="meta-llama/Llama-3.2-1B")
    parser.add_argument("--dataset_name", type=str, default="HuggingFaceH4/ultrachat_200k")
    parser.add_argument("--output_dir", type=str, default="./outputs/llama32_1b_zero2")
    parser.add_argument("--max_length", type=int, default=1024)

    parser.add_argument("--per_device_train_batch_size", type=int, default=1)
    parser.add_argument("--per_device_eval_batch_size", type=int, default=1)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=8)

    parser.add_argument("--learning_rate", type=float, default=2e-5)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--num_train_epochs", type=float, default=1.0)
    parser.add_argument("--warmup_ratio", type=float, default=0.03)

    parser.add_argument("--logging_steps", type=int, default=10)
    parser.add_argument("--save_steps", type=int, default=200)
    parser.add_argument("--eval_steps", type=int, default=200)

    parser.add_argument("--max_train_samples", type=int, default=None)
    parser.add_argument("--max_eval_samples", type=int, default=1000)

    parser.add_argument("--seed", type=int, default=42)
    return parser.parse_args()


def main():
    args = parse_args()
    set_seed(args.seed)

    hf_token = os.environ.get("HF_TOKEN", None)
    if hf_token is None:
        raise ValueError("HF_TOKEN is not set. You need access to the gated Llama repo.")

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    os.makedirs(args.output_dir, exist_ok=True)

    if is_main_process():
        print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        args.model_name,
        token=hf_token,
        use_fast=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if is_main_process():
        print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        token=hf_token,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
    )

    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    if is_main_process():
        print("Loading dataset...")
    raw_train = load_dataset(args.dataset_name, split="train_sft")
    raw_eval = load_dataset(args.dataset_name, split="test_sft")

    if args.max_train_samples is not None:
        raw_train = raw_train.select(range(min(args.max_train_samples, len(raw_train))))
    if args.max_eval_samples is not None:
        raw_eval = raw_eval.select(range(min(args.max_eval_samples, len(raw_eval))))

    def preprocess(example: Dict[str, Any]) -> Dict[str, Any]:
        text = format_messages_as_text(example["messages"], tokenizer.eos_token)
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=args.max_length,
            padding=False,
        )
        return tokenized

    remove_cols = raw_train.column_names

    train_dataset = raw_train.map(
        preprocess,
        remove_columns=remove_cols,
        desc="Tokenizing train",
    )

    eval_dataset = raw_eval.map(
        preprocess,
        remove_columns=remove_cols,
        desc="Tokenizing eval",
    )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    training_args = TrainingArguments(
        output_dir=args.output_dir,
        deepspeed="ds_zero2.json",

        per_device_train_batch_size=args.per_device_train_batch_size,
        per_device_eval_batch_size=args.per_device_eval_batch_size,
        gradient_accumulation_steps=args.gradient_accumulation_steps,

        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        warmup_ratio=args.warmup_ratio,
        num_train_epochs=args.num_train_epochs,

        lr_scheduler_type="cosine",

        logging_strategy="steps",
        logging_steps=args.logging_steps,

        eval_strategy="steps",
        eval_steps=args.eval_steps,

        save_strategy="steps",
        save_steps=args.save_steps,
        save_total_limit=2,

        bf16=bf16_ok,
        fp16=not bf16_ok,

        dataloader_num_workers=2,
        dataloader_pin_memory=True,

        gradient_checkpointing=True,
        report_to="none",

        remove_unused_columns=False,
        prediction_loss_only=True,
    )

    trainer = TokenCountingTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )

    if is_main_process():
        print("Starting training...")
    start_time = time.time()

    train_result = trainer.train()

    total_train_time = time.time() - start_time

    if is_main_process():
        metrics = train_result.metrics
        metrics["total_train_time_sec"] = total_train_time
        metrics["peak_gpu_mem_gb"] = (
            torch.cuda.max_memory_allocated() / (1024 ** 3)
            if torch.cuda.is_available() else 0.0
        )

        trainer.save_model()
        tokenizer.save_pretrained(args.output_dir)
        trainer.save_state()

        print("\n===== FINAL TRAIN METRICS =====")
        print(json.dumps(metrics, indent=2))

        eval_metrics = trainer.evaluate()
        print("\n===== FINAL EVAL METRICS =====")
        print(json.dumps(eval_metrics, indent=2))

        with open(os.path.join(args.output_dir, "final_metrics.json"), "w") as f:
            json.dump(
                {
                    "train_metrics": metrics,
                    "eval_metrics": eval_metrics,
                },
                f,
                indent=2,
            )

        print(f"\nDone. Outputs saved to: {args.output_dir}")


if __name__ == "__main__":
    main()

Overwriting train_zero2_llama.py


In [5]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Enter your HF token: ")

Enter your HF token: ··········


In [15]:
!python train_zero2_llama.py \
  --model_name meta-llama/Llama-3.2-1B \
  --dataset_name HuggingFaceH4/ultrachat_200k \
  --output_dir ./outputs/llama32_1b_zero2_smoke \
  --max_length 256 \
  --per_device_train_batch_size 1 \
  --per_device_eval_batch_size 1 \
  --gradient_accumulation_steps 1 \
  --learning_rate 2e-5 \
  --num_train_epochs 1 \
  --logging_steps 1 \
  --save_steps 50 \
  --eval_steps 50 \
  --max_train_samples 50 \
  --max_eval_samples 10

Loading tokenizer...
Loading model...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:00<00:00, 1698.87it/s, Materializing param=model.norm.weight]
Loading dataset...
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Starting training...
{'loss': '1.582', 'grad_norm': '41.53', 'learning_rate': '0', 'tokens_per_sec': '101.5', 'gpu_peak_mem_gb': '23.04', 'epoch': '0.02'}
{'loss': '2.769', 'grad_norm': '24.26', 'learning_rate': '1e-05', 'tokens_per_sec': '1251', 'gpu_peak_mem_gb': '25.34', 'epoch': '0.04'}
{'loss': '1.344', 'grad_norm': '23.14', 'learning_rate': '2e-05', 'tokens_per_sec': '1220', 'gpu_peak_mem_gb': '25.34', 'epoch': '0.06'}
{'loss': '2.654', 'grad_norm': '27.56', 'learning_rate': '1.998e-05', 'tokens_per_sec': '1212', 'gpu_peak_mem_gb': '25.34', 'epoch': '0.08'}
{'loss': '2.359', 'grad_norm': '22.55', 'learning_rate': '1.991e-05', 'tokens_per_sec': '1261', 'gpu_peak_mem_gb': '25.34', 'epoch': '0.1'}
{'l

In [17]:
!python train_zero2_llama.py \
  --model_name meta-llama/Llama-3.2-1B \
  --dataset_name HuggingFaceH4/ultrachat_200k \
  --output_dir ./outputs/llama32_1b_zero2 \
  --max_length 1024 \
  --per_device_train_batch_size 1 \
  --per_device_eval_batch_size 1 \
  --gradient_accumulation_steps 8 \
  --learning_rate 2e-5 \
  --num_train_epochs 1 \
  --logging_steps 10 \
  --save_steps 200 \
  --eval_steps 200 \
  --max_train_samples 20000 \
  --max_eval_samples 1000

Loading tokenizer...
Loading model...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:00<00:00, 2139.00it/s, Materializing param=model.norm.weight]
Loading dataset...
Tokenizing train: 100% 20000/20000 [01:16<00:00, 260.05 examples/s]
Tokenizing eval: 100% 1000/1000 [00:03<00:00, 264.51 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Starting training...
[RANK 0] Gradient accumulation steps mismatch: GradientAccumulationPlugin has 1, DeepSpeed config has 8. Using DeepSpeed's value.
{'loss': '1.579', 'grad_norm': '5.293', 'learning_rate': '2.4e-06', 'tokens_per_sec': '4833', 'gpu_peak_mem_gb': '25.34', 'epoch': '0.004'}
{'loss': '1.518', 'grad_norm': '4.665', 'learning_rate': '5.067e-06', 'tokens_per_sec': '5632', 'gpu_peak_mem_gb': '25.34', 'epoch': '0.008'}
{'loss': '1.479', 'grad_norm': '4.296', 'learning_rate': '7.733e-06', 'tokens_per_sec': '5896', 'gpu_peak_mem_gb': '25.34', 'epoch': '0.012'}
{'los

In [18]:
!ls -R ./outputs/llama32_1b_zero2 | head -200

./outputs/llama32_1b_zero2:
checkpoint-2400
checkpoint-2500
config.json
final_metrics.json
generation_config.json
model.safetensors
tokenizer_config.json
tokenizer.json
trainer_state.json
training_args.bin

./outputs/llama32_1b_zero2/checkpoint-2400:
config.json
generation_config.json
global_step2400
latest
model.safetensors
rng_state.pth
scheduler.pt
tokenizer_config.json
tokenizer.json
trainer_state.json
training_args.bin
zero_to_fp32.py

./outputs/llama32_1b_zero2/checkpoint-2400/global_step2400:
bf16_zero_pp_rank_0_mp_rank_00_optim_states.pt
mp_rank_00_model_states.pt

./outputs/llama32_1b_zero2/checkpoint-2500:
config.json
generation_config.json
global_step2500
latest
model.safetensors
rng_state.pth
scheduler.pt
tokenizer_config.json
tokenizer.json
trainer_state.json
training_args.bin
zero_to_fp32.py

./outputs/llama32_1b_zero2/checkpoint-2500/global_step2500:
bf16_zero_pp_rank_0_mp_rank_00_optim_states.pt
mp_rank_00_model_states.pt


In [19]:
import json
with open("./outputs/llama32_1b_zero2/final_metrics.json", "r") as f:
    metrics = json.load(f)
metrics

{'train_metrics': {'train_runtime': 4247.9999,
  'train_samples_per_second': 4.708,
  'train_steps_per_second': 0.589,
  'total_flos': 1.0196585663587942e+17,
  'train_loss': 1.4448070831298827,
  'gpu_peak_mem_gb': 25.338228225708008,
  'epoch': 1.0,
  'total_train_time_sec': 4249.576092720032,
  'peak_gpu_mem_gb': 25.338228225708008},
 'eval_metrics': {'eval_loss': 1.4053161144256592,
  'eval_runtime': 24.6267,
  'eval_samples_per_second': 40.606,
  'eval_steps_per_second': 40.606,
  'gpu_peak_mem_gb': 25.338228225708008,
  'epoch': 1.0}}

In [20]:
%%writefile export_for_vllm.py
import os
import argparse

import torch
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint_dir", type=str, required=True)
    parser.add_argument("--export_dir", type=str, required=True)
    parser.add_argument("--repo_id", type=str, default=None)
    parser.add_argument("--private", action="store_true")
    return parser.parse_args()


def main():
    args = parse_args()

    hf_token = os.environ.get("HF_TOKEN", None)
    if hf_token is None:
        raise ValueError("HF_TOKEN is not set.")

    os.makedirs(args.export_dir, exist_ok=True)

    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    print(f"Loading checkpoint from: {args.checkpoint_dir}")
    model = AutoModelForCausalLM.from_pretrained(
        args.checkpoint_dir,
        token=hf_token,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        device_map="cpu",
    )

    tokenizer = AutoTokenizer.from_pretrained(
        args.checkpoint_dir,
        token=hf_token,
        use_fast=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Saving clean export to: {args.export_dir}")
    model.save_pretrained(
        args.export_dir,
        safe_serialization=True,
        max_shard_size="5GB",
    )
    tokenizer.save_pretrained(args.export_dir)

    print("Local export complete.")

    if args.repo_id:
        print(f"Pushing to Hugging Face Hub: {args.repo_id}")
        api = HfApi(token=hf_token)
        api.create_repo(repo_id=args.repo_id, private=args.private, exist_ok=True)
        api.upload_folder(
            repo_id=args.repo_id,
            folder_path=args.export_dir,
            commit_message="Upload ZeRO-2 fine-tuned Llama 3.2 1B export for vLLM",
        )
        print("Push complete.")


if __name__ == "__main__":
    main()

Writing export_for_vllm.py


In [21]:
!python export_for_vllm.py \
  --checkpoint_dir ./outputs/llama32_1b_zero2 \
  --export_dir ./exports/llama32_1b_zero2_export

Loading checkpoint from: ./outputs/llama32_1b_zero2
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 147/147 [00:00<00:00, 1179.45it/s, Materializing param=model.norm.weight]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Saving clean export to: ./exports/llama32_1b_zero2_export
Writing model shards: 100% 1/1 [00:07<00:00,  7.38s/it]
Local export complete.


In [22]:
!zip -r llama32_outputs.zip ./outputs/llama32_1b_zero2

  adding: outputs/llama32_1b_zero2/ (stored 0%)
  adding: outputs/llama32_1b_zero2/generation_config.json (deflated 32%)
  adding: outputs/llama32_1b_zero2/training_args.bin (deflated 53%)
  adding: outputs/llama32_1b_zero2/tokenizer.json (deflated 85%)
  adding: outputs/llama32_1b_zero2/model.safetensors (deflated 21%)
  adding: outputs/llama32_1b_zero2/config.json (deflated 52%)
  adding: outputs/llama32_1b_zero2/trainer_state.json (deflated 80%)
  adding: outputs/llama32_1b_zero2/checkpoint-2500/ (stored 0%)
  adding: outputs/llama32_1b_zero2/checkpoint-2500/generation_config.json (deflated 32%)
  adding: outputs/llama32_1b_zero2/checkpoint-2500/training_args.bin (deflated 53%)
  adding: outputs/llama32_1b_zero2/checkpoint-2500/global_step2500/ (stored 0%)
  adding: outputs/llama32_1b_zero2/checkpoint-2500/global_step2500/mp_rank_00_model_states.pt (deflated 21%)
  adding: outputs/llama32_1b_zero2/checkpoint-2500/global_step2500/bf16_zero_pp_rank_0_mp_rank_00_optim_states.pt


zip e

In [26]:
!zip -r llama32_export.zip ./exports/llama32_1b_zero2_export

  adding: exports/llama32_1b_zero2_export/ (stored 0%)
  adding: exports/llama32_1b_zero2_export/generation_config.json (deflated 32%)
  adding: exports/llama32_1b_zero2_export/tokenizer.json (deflated 85%)
  adding: exports/llama32_1b_zero2_export/model.safetensors (deflated 21%)
  adding: exports/llama32_1b_zero2_export/config.json (deflated 52%)
  adding: exports/llama32_1b_zero2_export/tokenizer_config.json (deflated 49%)


In [27]:
from google.colab import files
#files.download("llama32_outputs.zip")
files.download("llama32_export.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
!python export_for_vllm.py \
  --checkpoint_dir ./outputs/llama32_1b_zero2 \
  --export_dir ./exports/llama32_1b_zero2_export \
  --repo_id robsol/llama32-1b-zero2

Loading checkpoint from: ./outputs/llama32_1b_zero2
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 147/147 [00:00<00:00, 3168.00it/s, Materializing param=model.norm.weight]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Saving clean export to: ./exports/llama32_1b_zero2_export
Writing model shards: 100% 1/1 [00:07<00:00,  7.23s/it]
Local export complete.
Pushing to Hugging Face Hub: robsol/llama32-1b-zero2
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
ht